In [1]:
import rebound
import reboundx

# Create a new REBOUND simulation object.
sim = rebound.Simulation()

# Set simulation units:
# - distance: AU
# - time: years
# - mass: solar masses
sim.units = ("AU", "yr", "Msun")

# Use the WHFast symplectic integrator.
sim.integrator = "whfast"

# Set the timestep in simulation time units (years here).
sim.dt = 1e-3

# Define a small list of test objects:
# Each tuple is (label, mass in Msun, SSE stellar type code).
objects = [
    ("MS",      1.0, 1),  # Main-sequence star
    ("GIANT",   1.0, 2),  # Giant
    ("HE",      1.0, 3),  # Helium star
    ("WD",      1.0, 4),  # White dwarf
    ("COMPACT", 1.0, 5),  # Compact remnant
]

# 1) Add particles to the REBOUND simulation first.
# Each particle is given the same mass, and placed at a different x-position
# so they do not overlap spatially.
for i, (label, m, ssetype) in enumerate(objects):
    sim.add(m=m, x=10.0 * i)

# 2) Attach REBOUNDx extras after particles have been added.
rebx = reboundx.Extras(sim)

# Load the stellar evolution operator based on SSE (Single Star Evolution).
sse = rebx.load_operator("stellar_evolution_sse")

# Add the SSE operator to the simulation so it is applied during integration.
rebx.add_operator(sse)

# Set scaling constants used by the SSE operator.
# These tell REBOUNDx how to interpret solar units in the current simulation units.
sse.params["sse_Msun"] = 1.0              # 1 solar mass in simulation mass units
sse.params["sse_Rsun"] = 1.0 / 215.032    # 1 solar radius in AU
sse.params["sse_Lsun"] = 1.0              # 1 solar luminosity in SSE luminosity units

# 3) Assign the stellar evolution type to each particle.
# REBOUNDx stores per-particle parameters in p.params, and sse_type is expected as a float.
for i, (label, m, ssetype) in enumerate(objects):
    sim.particles[i].params["sse_type"] = float(ssetype)

# Advance the simulation by one timestep.
# This gives the SSE operator a chance to initialize/update stellar properties.
sim.step()

# Print a small table of the resulting stellar properties.
print(f"{'Object':<10} {'M':>8} {'R [Rsun]':>12} {'L [Lsun]':>12}")
print("-" * 46)

for i, (label, m, ssetype) in enumerate(objects):
    p = sim.particles[i]

    # Convert particle radius to solar radii for display.
    # Note: this assumes p.r is in simulation length units.
    R_Rsun = p.r / 1.0 / 215.032

    # Read the stellar luminosity produced by the SSE operator.
    L_Lsun = p.params["sse_L"]

    # Print label, input mass, derived radius, and luminosity.
    print(f"{label:<10} {m:8.1f} {R_Rsun:12.6g} {L_Lsun:12.6g}")

Object            M     R [Rsun]     L [Lsun]
----------------------------------------------
MS              1.0  2.16269e-05            1
GIANT           1.0   0.00216269         2300
HE              1.0  4.32538e-06           50
WD              1.0  1.69727e-07        0.001
COMPACT         1.0  3.72847e-10            0
